# Structured Streaming on file events

A source system drops JSON files into `s3a://data/stream/landing/`. Spark's file source
treats each new file as an event: job 35 picks up only files it hasn't seen (the checkpoint
remembers them), writes Parquet partitioned by date and city, and archives what it read.
Doesn't need Kafka.

In [ ]:
import kyuubi_lab as lab          # client/kyuubi_lab.py (on PYTHONPATH in the Jupyter image)
import pandas as pd
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 200)
RID = lab.new_run_id()            # one id for every job in this notebook run
print("Kyuubi REST", lab.REST_URL, "| JDBC", f"{lab.JDBC_HOST}:{lab.JDBC_PORT}", "| user", lab.USER, "| run", RID)

In [ ]:
lab.produce(orders=25, run_id=RID + 'f', sink='s3', files=4)
lab.s3_ls('data', 'stream/landing/')

In [ ]:
r = lab.run('file-drop-stream', run_id=RID)
lab.log_tail(r, 'batch')

In [ ]:
silver = lab.sql("""SELECT city, event_type, count(*) AS events
                     FROM parquet.`s3a://data/stream/silver/order_events_by_city`
                     GROUP BY city, event_type""")
silver.pivot(index="city", columns="event_type", values="events").fillna(0).plot.bar(stacked=True, title="events by city");

## Run it again with no new files
The checkpoint already lists every file, so nothing is reprocessed.

In [ ]:
r = lab.run('file-drop-stream', run_id=RID)
lab.log_tail(r, 'batch')

## Live mode
Start the same job always-on (no `--available-now`), drop files while it runs, watch, then kill it.

In [ ]:
import time
bid = lab.submit_no_wait("file-drop-stream", RID, extra_args=[], conf={"spark.cores.max": "1"},
                         batch_name=f"nb-live-files-{RID}")
for _ in range(40):
    if lab.state(bid) == "RUNNING": break
    time.sleep(5)
lab.state(bid)

In [ ]:
for i in range(3):
    lab.produce(orders=5, run_id=f"{RID}L{i}", sink="s3", files=1)
    time.sleep(15)
print([l for l in lab.kyuubi().log(bid, 0, 2000).get("logRowSet", []) if "batch" in l][-6:])

In [ ]:
lab.kill(bid); time.sleep(3); lab.state(bid)